#This agent acts as the brain of my fraud detection model. It recommends the user the actions he/she should perform based on the risk of the transactions(HIGH/LOW).

#Base model for reasoning.

In [80]:
!pip install -q langchain langchain-groq

In [81]:
import os
from google.colab import userdata

os.environ['GROQ_API_KEY'] = userdata.get("GROQ_API_KEY")

In [82]:
from langchain.chat_models import init_chat_model
llm = init_chat_model("groq:openai/gpt-oss-120b") # Acts as the brain of our agent

#Creating the agent.

In [83]:
from langchain.agents import create_agent

#Loading the Finetuned model.

In [84]:
!pip install -q transformers peft

In [85]:
!pip install -q --upgrade torchao

In [86]:
from transformers import pipeline

repo_name = "shloksutar/fraud-detector"

generator = pipeline("text-generation", model=repo_name, device="cuda")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/224 [00:00<?, ?it/s]

#Creating required tool for Finetuned model.

In [87]:
from langchain.tools import tool

@tool
def analyze_transaction(
    transaction_type: str,
    amount: float,
    sender_balance_before: float,
    sender_balance_after: float,
    recipient_balance_before: float,
    recipient_balance_after: float
) -> str:
    """Analyze a financial transaction for fraud risk.
    Returns risk level (HIGH/LOW) with reasoning."""

    prompt = f"""Analyze this transaction for fraud risk:
- Type: {transaction_type}
- Amount: ${amount:,.2f}
- Sender Balance Before: ${sender_balance_before:,.2f}
- Sender Balance After: ${sender_balance_after:,.2f}
- Recipient Balance Before: ${recipient_balance_before:,.2f}
- Recipient Balance After: ${recipient_balance_after:,.2f}"""

    messages = [{"role": "user", "content": prompt}]
    output = generator(messages, max_new_tokens=150, return_full_text=False)[0]

    return output["generated_text"]

#Creating the final agent connected with the tool.

In [99]:
from langchain.agents import create_agent
system_prompt = """You are a fraud detection assistant.

When asked about a transaction, use the analyze_transaction tool to check for fraud risk.

Based on the analysis:
- If HIGH risk: Recommend blocking the transaction
- If LOW risk: Recommend approving the transaction

Always explain your reasoning clearly."""

agent = create_agent(
    model=llm,
    tools=[analyze_transaction],
    system_prompt=system_prompt,
)

print("Agent ready with the fraud tool attached.")

Agent ready with the fraud tool attached.


In [100]:
response = agent.invoke({
    "messages": [{"role": "user", "content": """Analyze this:
- Type: PAYMENT
- Amount: $150
- Sender balance: $5,000 before, $4,850 after
- Recipient balance: $1,000 before, $1,150 after"""}]
})

print(response["messages"][-1].content)

**Risk Assessment:** LOW  

**Explanation:**  
- The payment amount ($150) is modest relative to the sender’s balance (5 % of the $5,000 balance).  
- Both the sender’s and recipient’s balances change as expected: the sender’s balance drops by exactly the payment amount, and the recipient’s balance increases by the same amount.  
- The transaction type is a standard “PAYMENT,” with no unusual patterns (e.g., rapid large transfers, multiple recipients, or sudden spikes).  

**Recommendation:** Approve the transaction.
